In [0]:
pip install PyPDF2

In [0]:
# Databricks notebook source
# DBTITLE 1,Enhanced Web Scraping for ACA Policy Documents
"""
This is an improved version of step1_scrap_policy.py that:
1. Uses better source URLs (direct PDFs and government documents)
2. Extracts only main content (removes navigation/menus)
3. Cleans text for better LLM extraction
4. Writes to enhanced_scraped_text table
"""

# COMMAND ----------

# DBTITLE 1,Install Packages
# MAGIC %pip install requests beautifulsoup4 PyPDF2 lxml -q

# COMMAND ----------

# DBTITLE 1,Setup
dbutils.widgets.text("policy_catalog", 'gklick_catalog')
dbutils.widgets.text("policy_schema", 'aipolicyassistant')
dbutils.widgets.text("output_table", 'enhanced_scraped_text')

policy_catalog = dbutils.widgets.get("policy_catalog")
policy_schema = dbutils.widgets.get("policy_schema")
output_table = dbutils.widgets.get("output_table")

print(f"Output table: {policy_catalog}.{policy_schema}.{output_table}")

# COMMAND ----------

# DBTITLE 1,Import Libraries
import requests
from bs4 import BeautifulSoup
from PyPDF2 import PdfReader
from io import BytesIO
import re
from typing import Optional

# COMMAND ----------

# DBTITLE 1,Enhanced Scraping Functions

def clean_text(text: str) -> str:
    """
    Clean scraped text by removing navigation, excess whitespace, etc.
    """
    if not text:
        return ""
    
    # Remove common navigation phrases
    navigation_phrases = [
        "Skip to main content",
        "Toggle navigation",
        "Search Cornell",
        "Quick search by citation",
        "prev | next",
        "Support Us!",
        "Advertise",
        "Legal Information Institute",
        "Cookie Policy",
        "Accept Cookies",
        "Please help us improve",
        "No thank you",
        "Get the law",
        "Help out",
        "Join Lawyer Directory",
        "Create Promote Join"
    ]
    
    for phrase in navigation_phrases:
        text = text.replace(phrase, "")
    
    # Split into lines and filter
    lines = text.split('\n')
    clean_lines = []
    
    # Additional navigation keywords to filter
    nav_keywords = [
        'skip', 'toggle', 'menu', 'search', 'click here', 'home', 'about', 
        'contact', 'login', 'sign in', 'subscribe', 'share', 'print', 'download',
        'prev', 'next', 'go!', 'title', 'section', 'notes', 'quick search',
        'table of contents', 'breadcrumb', 'feedback', 'site map'
    ]
    
    for line in lines:
        line = line.strip()
        
        # Skip empty lines
        if not line:
            continue
        
        # Skip very short lines (likely navigation or headers)
        if len(line) < 50:
            line_lower = line.lower()
            # Skip if it contains navigation keywords
            if any(nav in line_lower for nav in nav_keywords):
                continue
            # Skip if it's mostly capital letters and short (headers)
            if len(line) < 30 and sum(c.isupper() for c in line if c.isalpha()) > len([c for c in line if c.isalpha()]) * 0.7:
                continue
        
        # Skip lines that are mostly punctuation or special characters
        if len(line) > 0 and sum(c in '|•→←©®™×' for c in line) > len(line) * 0.3:
            continue
        
        # Skip single-word lines
        if len(line.split()) == 1 and len(line) < 20:
            continue
        
        clean_lines.append(line)
    
    # Rejoin and clean up whitespace
    cleaned = '\n'.join(clean_lines)
    
    # Remove multiple blank lines (collapse to double newline)
    cleaned = re.sub(r'\n\s*\n\s*\n+', '\n\n', cleaned)
    
    # Remove excessive spaces
    cleaned = re.sub(r' +', ' ', cleaned)
    
    # Fix common spacing issues around punctuation
    cleaned = re.sub(r'\s+([.,;:])', r'\1', cleaned)  # Remove space before punctuation
    cleaned = re.sub(r'([.,;:])\s*\n', r'\1\n', cleaned)  # Ensure newline after punctuation at line end
    
    # Add proper spacing after section markers like (a), (1), etc.
    cleaned = re.sub(r'\(([a-z0-9]+)\)\s*\n', r'(\1)\n\n', cleaned)  # Double newline after section markers
    
    return cleaned.strip()


def scrape_pdf(url: str, timeout: int = 30) -> Optional[str]:
    """
    Scrape text from a PDF URL.
    """
    try:
        print(f"  Downloading PDF...")
        response = requests.get(url, timeout=timeout, headers={
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        })
        response.raise_for_status()
        
        print(f"  Extracting text from PDF...")
        pdf = PdfReader(BytesIO(response.content))
        
        text_parts = []
        for i, page in enumerate(pdf.pages):
            page_text = page.extract_text()
            if page_text:
                text_parts.append(page_text)
        
        full_text = "\n".join(text_parts)
        print(f"  Extracted {len(full_text)} characters from {len(pdf.pages)} pages")
        
        return clean_text(full_text)
        
    except Exception as e:
        print(f"  ✗ PDF extraction failed: {e}")
        return None


def scrape_html(url: str, timeout: int = 30) -> Optional[str]:
    """
    Scrape text from an HTML page, extracting only main content.
    """
    try:
        print(f"  Downloading HTML...")
        response = requests.get(url, timeout=timeout, headers={
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        })
        response.raise_for_status()
        
        print(f"  Parsing HTML...")
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Remove script, style, nav, header, footer, aside elements
        for element in soup.find_all(['script', 'style', 'nav', 'header', 'footer', 'aside', 'iframe']):
            element.decompose()
        
        # Remove common navigation/UI elements by class/id
        for element in soup.find_all(class_=re.compile(r'navigation|breadcrumb|sidebar|menu|banner|advertisement|cookie|modal', re.I)):
            element.decompose()
        
        for element in soup.find_all(id=re.compile(r'navigation|breadcrumb|sidebar|menu|banner|advertisement|cookie|modal', re.I)):
            element.decompose()
        
        # Remove role="navigation" elements
        for element in soup.find_all(attrs={'role': ['navigation', 'banner', 'complementary']}):
            element.decompose()
        
        # Try to find main content area (various strategies)
        main_content = None
        
        # Strategy 1: Look for specific content divs (Cornell LII)
        if 'cornell.edu' in url.lower() or 'law.cornell.edu' in url.lower():
            # Try multiple Cornell-specific selectors
            for selector in [
                ('div', {'id': 'toc-content'}),
                ('div', {'class': 'field-name-body'}),
                ('div', {'class': 'field-item'}),
                ('div', {'id': 'content'}),
                ('section', {'class': 'content'})
            ]:
                main_content = soup.find(selector[0], selector[1])
                if main_content:
                    break
        
        # Strategy 1b: Look for specific content divs (eCFR)
        if 'ecfr.gov' in url.lower():
            for selector in [
                ('div', {'id': 'content-wrapper'}),
                ('div', {'class': 'section-content'}),
                ('main', {}),
            ]:
                main_content = soup.find(selector[0], selector[1])
                if main_content:
                    break
        
        # Strategy 1c: Look for Federal Register content
        if 'federalregister.gov' in url.lower():
            for selector in [
                ('div', {'class': 'full-text'}),
                ('div', {'class': 'body-column'}),
                ('article', {})
            ]:
                main_content = soup.find(selector[0], selector[1])
                if main_content:
                    break
        
        # Strategy 2: Look for main/article tags
        if not main_content:
            main_content = soup.find('main')
        
        if not main_content:
            main_content = soup.find('article')
        
        # Strategy 3: Look for content divs
        if not main_content:
            main_content = soup.find('div', class_=re.compile(r'content|main|body|text', re.I))
        
        # Strategy 4: Look for the largest text block
        if not main_content:
            # Find all divs and pick the one with most text
            divs = soup.find_all('div')
            if divs:
                main_content = max(divs, key=lambda d: len(d.get_text(strip=True)))
        
        # Extract text with proper spacing
        # First, add line breaks after block elements to preserve structure
        if main_content:
            # Add newlines after block-level elements
            for tag in main_content.find_all(['p', 'div', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'li', 'tr', 'br']):
                tag.append('\n')
            # Use space as separator to preserve inline content
            text = main_content.get_text(separator=' ', strip=True)
        else:
            # Fallback: get all text (already removed nav/header/footer)
            for tag in soup.find_all(['p', 'div', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'li', 'tr', 'br']):
                tag.append('\n')
            text = soup.get_text(separator=' ', strip=True)
        
        print(f"  Extracted {len(text)} characters")
        
        return clean_text(text)
        
    except Exception as e:
        print(f"  ✗ HTML extraction failed: {e}")
        return None


def scrape_url(url: str) -> tuple[str, Optional[str], str]:
    """
    Scrape a URL and return (url, text, status).
    
    Returns:
        tuple: (url, scraped_text, status_message)
    """
    print(f"\nProcessing: {url}")
    
    if not url:
        return (url, None, "Empty URL")
    
    try:
        # Determine type and scrape
        if url.lower().endswith('.pdf'):
            text = scrape_pdf(url)
        else:
            text = scrape_html(url)
        
        if text and len(text) > 100:
            return (url, text, f"✓ Success ({len(text)} chars)")
        else:
            return (url, None, "✗ Insufficient content extracted")
            
    except Exception as e:
        return (url, None, f"✗ Error: {str(e)}")


# COMMAND ----------

# DBTITLE 1,Define High-Quality HRA Policy URLs
"""
These are URLs from the Authority.csv file plus additional high-quality sources.
All are authoritative government documents about HRAs, QSEHRA, ICHRA, and related policies.
"""

# URLs from Authority.csv file
QUALITY_URLS = [
    # ========================================================================
    # FROM AUTHORITY.CSV - STATUTES
    # ========================================================================
    
    # AuthN 01: 26 U.S.C. § 105 - Qualified Small Employer HRA (QSEHRA)
    "https://www.law.cornell.edu/uscode/text/26/105",
    
    # AuthN 02: 26 U.S.C. § 9831(d) - Creates QSEHRAs
    "https://www.law.cornell.edu/uscode/text/26/9831",
    
    # AuthN 04: 26 U.S.C. § 4980B - COBRA excise tax
    "https://www.law.cornell.edu/uscode/text/26/4980B",
    
    # AuthN 05: 29 U.S.C. §§ 1161–1168 - ERISA COBRA continuation
    "https://uscode.house.gov/view.xhtml?path=/prelim@title29/chapter18/subchapter1/subtitleB/part6",
    
    # AuthN 26: 26 USC § 5000A(f)(1) - Requirement to maintain MEC
    "https://www.law.cornell.edu/uscode/text/26/5000A",
    
    # ========================================================================
    # FROM AUTHORITY.CSV - FINAL RULES & REGULATIONS
    # ========================================================================
    
    # AuthN 03: 84 FR 28888 - Final Rule: HRAs & Account-Based Plans (ICHRA/EBHRA)
    "https://www.federalregister.gov/documents/2019/06/20/2019-12571/health-reimbursement-arrangements-and-other-account-based-group-health-plans",
    
    # AuthN 23: 26 CFR 54.4980H - Shared Responsibility for Employers
    "https://www.ecfr.gov/current/title-26/chapter-I/subchapter-D/part-54/section-54.4980H-0",
    
    # AuthN 06: 29 CFR 2590.606-1 - COBRA general notice framework
    "https://www.ecfr.gov/current/title-29/part-2590/subpart-G",
    
    # AuthN 07: 29 CFR 2590.606-4 - COBRA election notice requirements
    "https://www.ecfr.gov/current/title-29/part-2590/subpart-A/section-2590.606-4",
    
    # AuthN 08: 45 CFR 164.520 - HIPAA Notice of Privacy Practices
    "https://www.ecfr.gov/current/title-45/part-164/section-164.520",
    
    # AuthN 09: 26 CFR 54.9802-4 - ICHRA integration with individual coverage
    "https://www.ecfr.gov/current/title-26/part-54/section-54.9802-4",
    
    # AuthN 10: 26 CFR 54.9831-1(c)(3)(viii) - Excepted Benefit HRA (EBHRA)
    "https://www.ecfr.gov/current/title-26/part-54/section-54.9831-1",
    
    # AuthN 11: 26 CFR 54.9815-2711(d) - Integration rules for account-based plans
    "https://www.ecfr.gov/current/title-26/part-54/section-54.9815-2711",
    
    # AuthN 12: 45 CFR 146.123 - HHS parallel provisions
    "https://www.ecfr.gov/current/title-45/part-146/section-146.123",
    
    # AuthN 24: 26 CFR 31.3401(c)-1 - Defining Employee
    "https://www.ecfr.gov/current/title-26/section-31.3401(c)-1#p-31.3401(c)-1(b)",
    
    # AuthN 25: 26 CFR 31.3121(d)-1(c) - Common law standard for employer
    "https://www.ecfr.gov/current/title-26/section-31.3121(d)-1#p-31.3121(d)-1(c)",
    
    # ========================================================================
    # FROM AUTHORITY.CSV - IRS NOTICES, REVENUE PROCEDURES, RULINGS (PDFs)
    # ========================================================================
    
    # AuthN 13: Rev. Proc. 2024-35 - 2025 ACA affordability = 9.02%
    "https://www.irs.gov/pub/irs-drop/rp-24-35.pdf",
    
    # AuthN 14: IRS Publication 974 - Premium Tax Credit guidance
    "https://www.irs.gov/pub/irs-pdf/p974.pdf",
    
    # AuthN 15: Rev. Proc. 2024-25 - EBHRA 2025 cap = $2,150
    "https://www.irs.gov/pub/irs-drop/rp-24-25.pdf",
    
    # AuthN 16: Rev. Proc. 2024-40 - QSEHRA 2025 caps
    "https://www.irs.gov/pub/irs-drop/rp-24-40.pdf",
    
    # AuthN 17: IRS Notice 2002-45 - Foundational HRA rules
    "https://www.irs.gov/pub/irs-drop/n-02-45.pdf",
    
    # AuthN 18: Rev. Rul. 2002-41 - HRA integration guidance
    "https://www.irs.gov/pub/irs-irbs/irb02-28.pdf",
    
    # AuthN 19: IRS Notice 2017-67 - QSEHRA operational guidance
    "https://www.irs.gov/pub/irs-drop/n-17-67.pdf",
    
    # ========================================================================
    # FROM AUTHORITY.CSV - EXECUTIVE ORDERS
    # ========================================================================
    
    # AuthN 20: EO 13813 - Promoting Healthcare Choice
    "https://www.federalregister.gov/documents/2017/10/17/2017-22677/promoting-healthcare-choice-and-competition-across-the-united-states",
    
    # AuthN 21: EO 14009 - Strengthening Medicaid and ACA
    "https://www.federalregister.gov/documents/2021/02/02/2021-02252/strengthening-medicaid-and-the-affordable-care-act",
    
    # AuthN 22: EO 14148 - Initial Rescissions
    "https://www.federalregister.gov/documents/2025/01/28/2025-01901/initial-rescissions-of-harmful-executive-orders-and-actions",
    
    # ========================================================================
    # ADDITIONAL HIGH-QUALITY SOURCES (not in Authority.csv)
    # ========================================================================
    
    # DOL ACA Implementation FAQs
    "https://www.dol.gov/sites/dolgov/files/EBSA/about-ebsa/our-activities/resource-center/faqs/aca-part-43.pdf",
    "https://www.dol.gov/sites/dolgov/files/EBSA/about-ebsa/our-activities/resource-center/faqs/aca-part-36.pdf",
    
    # IRS Employer Health Care Arrangements page
    "https://www.irs.gov/affordable-care-act/employer-health-care-arrangements",
    
    # GovInfo versions (better for some statutes)
    "https://www.govinfo.gov/content/pkg/USCODE-2021-title26/html/USCODE-2021-title26-subtitleA-chap1-subchapB-partIII-sec105.htm",
    "https://www.govinfo.gov/content/pkg/USCODE-2021-title26/html/USCODE-2021-title26-subtitleK-chap100-subchapB-sec9831.htm",
]

print(f"Defined {len(QUALITY_URLS)} authoritative source URLs from Authority.csv")

# COMMAND ----------

# DBTITLE 1,Display URLs to be Scraped
print("=" * 80)
print("URLs TO BE SCRAPED")
print("=" * 80)

for i, url in enumerate(QUALITY_URLS, 1):
    url_type = "PDF" if url.endswith('.pdf') else "HTML"
    print(f"{i:2}. [{url_type:4}] {url}")

print(f"\nTotal: {len(QUALITY_URLS)} URLs")

# COMMAND ----------

# DBTITLE 1,Scrape All URLs
print("=" * 80)
print("STARTING SCRAPING")
print("=" * 80)

results = []

for i, url in enumerate(QUALITY_URLS, 1):
    print(f"\n[{i}/{len(QUALITY_URLS)}]", end=" ")
    
    url, text, status = scrape_url(url)
    
    results.append({
        'link': url,
        'scraped_text': text if text else "",
        'status': status,
        'text_length': len(text) if text else 0
    })
    
    print(f"  {status}")

print("\n" + "=" * 80)
print("SCRAPING COMPLETE")
print("=" * 80)

# COMMAND ----------

# DBTITLE 1,Show Scraping Results Summary
from pyspark.sql import Row

# Create DataFrame
results_df = spark.createDataFrame([Row(**r) for r in results])

# Summary statistics
total = len(results)
successful = len([r for r in results if r['text_length'] > 100])
failed = total - successful
success_rate = (successful / total * 100) if total > 0 else 0

print(f"\n📊 SCRAPING SUMMARY")
print(f"  Total URLs: {total}")
print(f"  ✓ Successful: {successful}")
print(f"  ✗ Failed: {failed}")
print(f"  Success Rate: {success_rate:.1f}%")
print()

# Show detailed results
print("Detailed Results:")
display(results_df.select("link", "status", "text_length"))

# COMMAND ----------

# DBTITLE 1,Check Content Quality
print("\n" + "=" * 80)
print("CONTENT QUALITY CHECK")
print("=" * 80)

from pyspark.sql import functions as F

quality_check = results_df.select(
    "link",
    "text_length",
    F.when(F.lower("scraped_text").contains("qsehra"), "QSEHRA")
     .when(F.lower("scraped_text").contains("ichra"), "ICHRA")
     .when(F.lower("scraped_text").contains("health reimbursement"), "HRA")
     .when(F.lower("scraped_text").contains("section 105"), "Section 105")
     .when(F.lower("scraped_text").contains("section 9831"), "Section 9831")
     .otherwise("Other").alias("content_type"),
    F.when(F.lower("scraped_text").contains("employer"), True).otherwise(False).alias("has_employer"),
    F.when(F.lower("scraped_text").contains("eligible"), True).otherwise(False).alias("has_eligibility"),
    F.when(F.lower("scraped_text").contains("requirement"), True).otherwise(False).alias("has_requirements"),
    F.when(F.lower("scraped_text").contains("contribution") | F.lower("scraped_text").contains("limit"), True).otherwise(False).alias("has_limits")
)

display(quality_check)

# Summary
hra_content = results_df.filter(
    F.lower("scraped_text").contains("qsehra") | 
    F.lower("scraped_text").contains("ichra") |
    F.lower("scraped_text").contains("health reimbursement")
).count()

good_length = results_df.filter(F.col("text_length") > 1000).count()

print(f"\n✓ Documents with HRA content: {hra_content}/{total} ({hra_content/total*100:.1f}%)")
print(f"✓ Documents with good length (>1000 chars): {good_length}/{total} ({good_length/total*100:.1f}%)")

# COMMAND ----------

# DBTITLE 1,Sample Content Preview
print("\n" + "=" * 80)
print("SAMPLE CONTENT PREVIEW")
print("=" * 80)

# Show a sample of the best content
best_doc = results_df.filter(F.col("text_length") > 1000).orderBy(F.desc("text_length")).first()

if best_doc:
    print(f"\nDocument: {best_doc.link}")
    print(f"Length: {best_doc.text_length:,} characters")
    print(f"\nFirst 1000 characters:")
    print("=" * 80)
    print(best_doc.scraped_text[:1000])
    print("=" * 80)
    
    # Check for key entities
    text_lower = best_doc.scraped_text.lower()
    print("\nKey Content Indicators:")
    print(f"  ✓ Contains 'QSEHRA': {('qsehra' in text_lower)}")
    print(f"  ✓ Contains 'ICHRA': {('ichra' in text_lower)}")
    print(f"  ✓ Contains 'employer': {('employer' in text_lower)}")
    print(f"  ✓ Contains 'eligible': {('eligible' in text_lower)}")
    print(f"  ✓ Contains 'requirement': {('requirement' in text_lower)}")
    print(f"  ✓ Contains dollar amounts: {bool(re.search(r'\$[\d,]+', best_doc.scraped_text))}")
    print(f"  ✓ Contains citations: {bool(re.search(r'section \d+|§\s*\d+|26 usc', text_lower))}")

# COMMAND ----------

# DBTITLE 1,Save to Enhanced Scraped Text Table
print("\n" + "=" * 80)
print("SAVING TO DATABASE")
print("=" * 80)

# Keep only successful scrapes for the output table
output_df = results_df.filter(F.col("text_length") > 100).select("link", "scraped_text")

output_table_name = f"`{policy_catalog}`.`{policy_schema}`.{output_table}"

# Save
output_df.write.mode("overwrite").saveAsTable(output_table_name)

saved_count = output_df.count()
print(f"✓ Saved {saved_count} documents to {output_table_name}")

# Verify
print(f"\nVerifying saved data...")
saved_df = spark.table(output_table_name)
print(f"✓ Verified: {saved_df.count()} rows in table")

# COMMAND ----------

# DBTITLE 1,Final Summary and Next Steps
print("\n" + "=" * 80)
print("🎉 ENHANCED SCRAPING COMPLETE!")
print("=" * 80)

print(f"""
✓ Scraped {total} URLs
✓ Saved {saved_count} high-quality documents
✓ Data stored in: {output_table_name}

CONTENT QUALITY:
  • Documents with HRA content: {hra_content} ({hra_content/total*100:.0f}%)
  • Documents with good length: {good_length} ({good_length/total*100:.0f}%)
  • Average text length: {results_df.agg(F.avg("text_length")).collect()[0][0]:,.0f} characters

NEXT STEPS:
1. Review sample content above to verify quality
2. Run entity extraction: step2_extract_entities_llm.py
   - Update to use table: {output_table_name}
   - Change line: scraped_df = spark.table("{output_table_name}")
3. Extraction should now find many more entities!

IMPROVEMENTS OVER ORIGINAL:
  ✓ Better source URLs (direct PDFs and government docs)
  ✓ Clean text extraction (navigation removed)
  ✓ Main content only (not website menus)
  ✓ Higher quality for LLM processing
""")

print("=" * 80)



In [0]:
%sql
   SELECT LEFT(scraped_text, 2000) as preview
   FROM gklick_catalog.aipolicyassistant.enhanced_scraped_text
   LIMIT 1